# Mission 3 — 기존 가중치 모델의 고정 임계값 0.5 평가 (재학습 없음)

현재 Drive의 `turn_chunk_50pct_seed42_weighted/best.pt`를 읽어 평가합니다.
Colab에 이 노트북 하나를 업로드하고 GPU 런타임에서 위부터 실행하세요.
**원래 학습 모델과 last.pt는 변경하지 않습니다.**

1. 기존과 같은 데이터 스냅샷에서 내부 검증 점수를 계산합니다.
2. 모든 증상에 동일한 고정 임계값 `0.5`를 적용합니다.
3. 공식 Validation의 지표와 증상별 오탐·미탐을 저장합니다.

멘토 지침에 따라 전역 임계값 탐색과 증상별 임계값 튜닝은 수행하지 않습니다.

결과는 기존 Drive 실행 폴더 안의 `threshold_review` 하위 폴더에 저장됩니다.


## 1. 패키지 준비
Colab에 제공된 GPU용 PyTorch는 그대로 사용합니다. CPU용 PyTorch를 새로 설치하지 않습니다.

In [ ]:
import subprocess
import sys

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "--quiet", "--no-compile",
    "transformers==5.17.0", "pandas>=2.0", "tqdm>=4.66", "matplotlib>=3.8",
])
print("학습 패키지 준비 완료")

## 2. Google Drive 연결

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 3. 경로와 학습 설정
아래 데이터 경로는 보내주신 스크린샷에 맞췄습니다. 폴더명이 같으면 바꿀 필요 없습니다.

같은 실험을 이어갈 때 `RUN_NAME`, 데이터 선택 비율, seed, batch size, 청크 설정, 학습률을 유지하세요.
`EPOCHS`는 총 목표 epoch이므로 1 → 3으로 늘리면 완료한 1 epoch 이후부터 3 epoch까지 진행합니다.
다른 설정으로 새로 실험하려면 **새 RUN_NAME**을 지정하세요.

In [ ]:
from pathlib import Path

MODE = "evaluate"  # "train": 학습/재개 후 평가, "evaluate": 저장된 best.pt만 평가
DATA_ROOT = Path("/content/drive/MyDrive/대학부 데이터")
TRAIN_LABEL_DIR = DATA_ROOT / "Training" / "2.라벨링데이터" / "TL_서울_구급"
VALIDATION_LABEL_DIR = DATA_ROOT / "Validation" / "2.라벨링데이터" / "VL_서울_구급"

RUN_NAME = "turn_chunk_50pct_seed42_weighted"
DRIVE_RUN_DIR = Path("/content/drive/MyDrive/DCC_MISSION3") / RUN_NAME
LOCAL_WORK_DIR = Path("/content/dcc_mission3") / RUN_NAME

MODEL_NAME = "beomi/kcbert-base"
SAMPLE_RATIO = 0.50
MAX_TRAIN_SAMPLES = None  # 빠른 실행 확인: SAMPLE_RATIO=None, MAX_TRAIN_SAMPLES=10
INTERNAL_VAL_RATIO = 0.10
SEED = 42
EPOCHS = 1
BATCH_SIZE = 2  # 청크 개수가 아닌 통화 수
LEARNING_RATE = 2e-5
MAX_LENGTH = 256
LONG_TURN_OVERLAP = 64  # 긴 턴을 토큰으로 나눌 때만 사용. 일반 청크는 마지막 턴을 겹칩니다.
THRESHOLD = 0.50  # 멘토 지침: 모든 증상에 고정 적용(튜닝하지 않음)
USE_POS_WEIGHT = True
POS_WEIGHT_SCALE = 1.0  # 1: 학습 음성/양성 비율, 0: 가중치 1
POS_WEIGHT_MAX = None  # 내부 검증으로 조정할 상한. 예: 10.0

SAVE_EVERY_STEPS = 200
SAVE_EVERY_MINUTES = 10
AUTO_RESUME = True
REQUIRE_GPU = True
SESSION_MAX_STEPS = None  # 필요하면 한 세션에서 처리할 step 수를 지정. 중간 저장 후 종료합니다.
EVAL_CHUNK_BATCH = 16
OFFICIAL_VALIDATION_MAX_SAMPLES = None  # 기본: 공식 Validation 전체 평가

TARGET_SYMPTOMS = ["고열", "구토", "두통", "복통", "어지러움", "열상", "오심", "전신쇠약", "호흡곤란"]
CHUNK_STRATEGY = "dialogue_turn_overlap_v1"
THRESHOLD_POLICY = "fixed_0.5"


## 4. 환경 확인

In [ ]:
import gc
import hashlib
import json
import math
import os
import random
import shutil
import time
import zipfile
from typing import Any, Sequence

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
import transformers
from transformers import AutoModel, AutoTokenizer

def check_settings():
    if MODE not in {"train", "evaluate"}:
        raise ValueError("MODE는 train 또는 evaluate여야 합니다.")
    if SAMPLE_RATIO is not None and MAX_TRAIN_SAMPLES is not None:
        raise ValueError("SAMPLE_RATIO와 MAX_TRAIN_SAMPLES는 동시에 지정할 수 없습니다.")
    if SAMPLE_RATIO is not None and not 0 < SAMPLE_RATIO <= 1:
        raise ValueError("SAMPLE_RATIO는 0보다 크고 1 이하여야 합니다.")
    if MAX_TRAIN_SAMPLES is not None and MAX_TRAIN_SAMPLES < 2:
        raise ValueError("MAX_TRAIN_SAMPLES는 2 이상이어야 합니다.")
    if EPOCHS < 1 or BATCH_SIZE < 1 or EVAL_CHUNK_BATCH < 1:
        raise ValueError("EPOCHS, BATCH_SIZE, EVAL_CHUNK_BATCH는 1 이상이어야 합니다.")
    if not 0 < INTERNAL_VAL_RATIO < 1:
        raise ValueError("검증 비율이 올바르지 않습니다.")
    if THRESHOLD != 0.5 or THRESHOLD_POLICY != "fixed_0.5":
        raise ValueError("판정 임계값은 모든 증상에 0.5로 고정해야 합니다.")
    if SAVE_EVERY_STEPS < 1 or SAVE_EVERY_MINUTES <= 0:
        raise ValueError("중간 저장 간격은 양수여야 합니다.")
    if SESSION_MAX_STEPS is not None and SESSION_MAX_STEPS < 1:
        raise ValueError("SESSION_MAX_STEPS는 1 이상이어야 합니다.")
    if OFFICIAL_VALIDATION_MAX_SAMPLES is not None and OFFICIAL_VALIDATION_MAX_SAMPLES < 1:
        raise ValueError("OFFICIAL_VALIDATION_MAX_SAMPLES는 1 이상이어야 합니다.")
    if REQUIRE_GPU and not torch.cuda.is_available():
        raise RuntimeError("GPU가 연결되지 않았습니다. 런타임 유형을 GPU로 변경한 뒤 다시 실행하세요.")

check_settings()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__, "/ transformers:", transformers.__version__)
print("학습 장치:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
print("데이터:", DATA_ROOT)
print("Drive 결과 폴더:", DRIVE_RUN_DIR)
DRIVE_RUN_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_WORK_DIR.mkdir(parents=True, exist_ok=True)

## 5. 발화 경계를 고려한 청크와 평가 지표
인접 턴 전체가 본문 한도 안에 들어가면 같은 청크에 함께 담습니다.
너무 긴 턴은 토큰 overlap으로 나누고, 서로 다른 턴을 함께 담을 수 없으면 경계 문맥 청크를 추가합니다.
원문 토큰을 버리지 않습니다.

In [ ]:
def build_dialogue_turns(utterances: list[dict[str, Any]]) -> list[str]:
    """입력 순서를 유지하며 같은 화자의 연속 발화를 하나의 턴으로 묶는다.

    화자 정보가 없는 발화는 독립 턴으로 유지한다. 화자 번호의 역할이나
    물음표에 의존하지 않으므로 인접한 질문·답변 문맥을 함께 보존한다.
    """
    turns = []
    previous_speaker = None
    for utterance in utterances:
        text = utterance.get("text", "").strip()
        if not text:
            continue
        speaker = utterance.get("speaker")
        if turns and speaker is not None and speaker == previous_speaker:
            turns[-1] += " " + text
        else:
            turns.append(text)
        previous_speaker = speaker
    return turns

def build_dialogue_token_chunks(
    tokenizer,
    utterances: list[dict[str, Any]],
    max_length: int = 256,
    chunk_overlap: int = 64,
) -> list[list[int]]:
    """턴 경계에서 청크를 나누고 마지막 턴을 다음 청크에 겹쳐 넣는다.

    인접한 두 턴이 본문 한도 안에 들어가면 반드시 하나 이상의 청크에서
    함께 나타난다. 한도를 넘는 턴은 토큰 overlap으로 분할한다. 서로 다른
    턴을 함께 담을 수 없으면 앞 턴의 끝과 뒤 턴의 시작을 담은 경계 청크를
    추가한다. 모든 원문 토큰을 보존하며 청크 길이는 본문 한도를 넘지 않는다.
    chunk_overlap은 긴 턴 내부 분할에만 사용한다. 정상 턴은 한 턴을 겹친다.
    """
    capacity = max_length - tokenizer.num_special_tokens_to_add(pair=False)
    if capacity <= 0:
        raise ValueError("max_length는 특수 토큰 수보다 커야 합니다.")
    if not 0 <= chunk_overlap < capacity:
        raise ValueError("chunk_overlap은 0 이상이며 본문 토큰 한도보다 작아야 합니다.")

    pieces = []
    for turn_index, text in enumerate(build_dialogue_turns(utterances)):
        tokens = tokenizer(
            text, add_special_tokens=False, truncation=False,
            return_attention_mask=False, verbose=False,
        )["input_ids"]
        start = 0
        while start < len(tokens):
            pieces.append((turn_index, tokens[start : start + capacity]))
            if start + capacity >= len(tokens):
                break
            start += capacity - chunk_overlap
    if not pieces:
        return [[]]

    chunks = []
    current = []
    previous_tokens = []
    previous_turn = None
    for turn_index, tokens in pieces:
        if len(current) + len(tokens) <= capacity:
            current = current + tokens
        else:
            chunks.append(current)
            if len(previous_tokens) + len(tokens) <= capacity:
                current = previous_tokens + tokens
            else:
                # 두 턴 전체가 안 들어가도 질문의 끝과 답변의 시작을 보존.
                # 같은 긴 턴 내부에서는 이미 token overlap을 적용했다.
                if previous_turn != turn_index and capacity >= 2:
                    left = min(len(previous_tokens), capacity // 2)
                    right = min(len(tokens), capacity - left)
                    left = min(len(previous_tokens), capacity - right)
                    chunks.append(previous_tokens[-left:] + tokens[:right])
                current = tokens.copy()
        previous_tokens = tokens
        previous_turn = turn_index
    chunks.append(current)
    return chunks

def _pad_token_chunks(tokenizer, raw_chunks, max_length):
    """본문 토큰 청크에 특수 토큰을 추가하고 같은 길이로 padding."""
    # transformers 5.x의 일부 slow BertTokenizer에서는 예전 공개 메서드인
    # build_inputs_with_special_tokens가 제공되지 않는다. KcBERT는 단일 문장을
    # [CLS] 본문 [SEP] 순서로 구성하므로 해당 버전에서는 토큰 ID를 직접 붙인다.
    build_with_specials = getattr(
        tokenizer,
        "build_inputs_with_special_tokens",
        None,
    )
    if callable(build_with_specials):
        chunks_with_specials = [build_with_specials(chunk) for chunk in raw_chunks]
    else:
        cls_token_id = tokenizer.cls_token_id
        sep_token_id = tokenizer.sep_token_id
        if cls_token_id is None or sep_token_id is None:
            raise ValueError("토크나이저에 CLS/SEP 특수 토큰 ID가 없습니다.")
        chunks_with_specials = [
            [cls_token_id, *chunk, sep_token_id]
            for chunk in raw_chunks
        ]
    encoded = tokenizer.pad(
        {"input_ids": chunks_with_specials},
        padding="max_length",
        max_length=max_length,
        return_attention_mask=True,
        return_tensors="pt",
    )
    return {
        "input_ids": encoded["input_ids"],
        "attention_mask": encoded["attention_mask"],
    }

def tokenize_dialogue_chunks(
    tokenizer,
    utterances: list[dict[str, Any]],
    max_length: int = 256,
    chunk_overlap: int = 64,
) -> dict[str, torch.Tensor]:
    """발화 경계를 고려한 청크를 모델 입력 텐서로 반환."""
    chunks = build_dialogue_token_chunks(
        tokenizer, utterances, max_length=max_length, chunk_overlap=chunk_overlap,
    )
    return _pad_token_chunks(tokenizer, chunks, max_length)

In [ ]:
def _divide(numerator, denominator):
    """0으로 나누는 경우 0을 반환하는 나눗셈."""
    return np.divide(
        numerator,
        denominator,
        out=np.zeros_like(np.asarray(numerator, dtype=float)),
        where=np.asarray(denominator) != 0,
    )

def compute_multilabel_metrics(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    label_names: Sequence[str],
) -> dict[str, object]:
    """Binary multi-label 예측의 전체/라벨별 지표를 계산한다.

    scikit-learn 없이 NumPy만 사용해 학습 환경의 의존성을 늘리지 않는다.
    모든 입력은 ``(샘플 수, 라벨 수)`` 형태의 0/1 배열이어야 한다.
    """
    y_true = np.asarray(y_true, dtype=np.int8)
    y_pred = np.asarray(y_pred, dtype=np.int8)
    if y_true.shape != y_pred.shape or y_true.ndim != 2:
        raise ValueError(
            "y_true와 y_pred는 동일한 (샘플 수, 라벨 수) 2차원 배열이어야 합니다."
        )
    if y_true.shape[1] != len(label_names):
        raise ValueError("label_names 수가 배열의 라벨 수와 다릅니다.")
    if y_true.shape[0] == 0:
        raise ValueError("평가할 샘플이 없습니다.")

    tp = np.sum((y_true == 1) & (y_pred == 1), axis=0)
    fp = np.sum((y_true == 0) & (y_pred == 1), axis=0)
    fn = np.sum((y_true == 1) & (y_pred == 0), axis=0)
    tn = np.sum((y_true == 0) & (y_pred == 0), axis=0)

    precision = _divide(tp, tp + fp)
    recall = _divide(tp, tp + fn)
    f1 = _divide(2 * precision * recall, precision + recall)
    specificity = _divide(tn, tn + fp)
    support = tp + fn

    micro_tp = int(tp.sum())
    micro_fp = int(fp.sum())
    micro_fn = int(fn.sum())
    micro_precision = float(_divide(micro_tp, micro_tp + micro_fp))
    micro_recall = float(_divide(micro_tp, micro_tp + micro_fn))
    micro_f1 = float(
        _divide(
            2 * micro_precision * micro_recall,
            micro_precision + micro_recall,
        )
    )

    sample_tp = np.sum((y_true == 1) & (y_pred == 1), axis=1)
    sample_predicted = np.sum(y_pred == 1, axis=1)
    sample_actual = np.sum(y_true == 1, axis=1)
    sample_precision = _divide(sample_tp, sample_predicted)
    sample_recall = _divide(sample_tp, sample_actual)
    sample_f1 = _divide(
        2 * sample_precision * sample_recall,
        sample_precision + sample_recall,
    )

    support_total = int(support.sum())
    weighted_f1 = (
        float(np.average(f1, weights=support)) if support_total else 0.0
    )

    per_label = {
        name: {
            "precision": float(precision[index]),
            "recall": float(recall[index]),
            "f1": float(f1[index]),
            "specificity": float(specificity[index]),
            "support": int(support[index]),
            "tp": int(tp[index]),
            "fp": int(fp[index]),
            "fn": int(fn[index]),
            "tn": int(tn[index]),
        }
        for index, name in enumerate(label_names)
    }

    return {
        "num_samples": int(y_true.shape[0]),
        "num_labels": int(y_true.shape[1]),
        "macro_precision": float(np.mean(precision)),
        "macro_recall": float(np.mean(recall)),
        "macro_f1": float(np.mean(f1)),
        "micro_precision": micro_precision,
        "micro_recall": micro_recall,
        "micro_f1": micro_f1,
        "weighted_f1": weighted_f1,
        "samples_precision": float(np.mean(sample_precision)),
        "samples_recall": float(np.mean(sample_recall)),
        "samples_f1": float(np.mean(sample_f1)),
        "subset_accuracy": float(np.mean(np.all(y_true == y_pred, axis=1))),
        "hamming_loss": float(np.mean(y_true != y_pred)),
        "label_accuracy": float(np.mean(y_true == y_pred)),
        "per_label": per_label,
    }

def print_metrics(metrics: dict[str, object]) -> None:
    """주요 지표와 라벨별 지표를 콘솔에 출력."""
    print("\n[전체 지표]")
    keys = [
        "macro_f1",
        "micro_f1",
        "weighted_f1",
        "samples_f1",
        "macro_precision",
        "macro_recall",
        "subset_accuracy",
        "hamming_loss",
        "label_accuracy",
    ]
    for key in keys:
        print(f"  {key:18s}: {metrics[key]:.4f}")

    print("\n[라벨별 지표]")
    print(f"  {'label':8s} {'precision':>9s} {'recall':>9s} {'f1':>9s} {'support':>9s}")
    for label, values in metrics["per_label"].items():
        print(
            f"  {label:8s} {values['precision']:9.4f} "
            f"{values['recall']:9.4f} {values['f1']:9.4f} "
            f"{values['support']:9d}"
        )

## 6. 데이터 스냅샷 준비
처음에는 Drive에서 선택한 JSON을 읽어 로컬 압축 파일을 만들고, 완료한 압축 파일만 Drive에 복사합니다.
이 단계는 파일 수에 따라 시간이 걸립니다. 원천 음성 파일은 읽거나 복사하지 않습니다.
이후 재연결 때는 이 압축 파일을 사용하므로 같은 데이터와 분할을 유지합니다.
데이터를 수정해 새 실험을 하려면 RUN_NAME도 새로 지정하세요.

In [ ]:
def dataset_selection_config():
    return {
        "sample_ratio": SAMPLE_RATIO, "max_train_samples": MAX_TRAIN_SAMPLES,
        "internal_val_ratio": INTERNAL_VAL_RATIO, "seed": SEED,
        "train_source": str(TRAIN_LABEL_DIR), "validation_source": str(VALIDATION_LABEL_DIR),
        "official_validation_max_samples": OFFICIAL_VALIDATION_MAX_SAMPLES,
    }

def publish_file(local_file, destination):
    """완료된 로컬 파일을 Drive 임시 파일로 복사한 뒤 기존 파일을 교체."""
    destination.parent.mkdir(parents=True, exist_ok=True)
    pending = destination.with_name(destination.name + ".pending")
    shutil.copyfile(local_file, pending)
    os.replace(pending, destination)

def write_json_result(data, name):
    temporary = LOCAL_WORK_DIR / name
    temporary.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")
    publish_file(temporary, DRIVE_RUN_DIR / name)

def prepare_data_snapshot():
    archive = DRIVE_RUN_DIR / "data_snapshot.zip"
    local_archive = LOCAL_WORK_DIR / "data_snapshot.zip"
    if archive.is_file():
        print("저장된 데이터 스냅샷을 사용합니다.")
        shutil.copyfile(archive, local_archive)
        with zipfile.ZipFile(local_archive) as packed:
            manifest = json.loads(packed.read("manifest.json"))
            if manifest["selection_config"] != dataset_selection_config():
                raise ValueError("기존 스냅샷과 데이터 선택 설정이 다릅니다. RUN_NAME을 새로 지정하세요.")
    else:
        for path in [TRAIN_LABEL_DIR, VALIDATION_LABEL_DIR]:
            if not path.is_dir():
                raise FileNotFoundError(f"라벨 폴더가 없습니다: {path}. 설정 셀의 경로를 확인하세요.")
        files = sorted(TRAIN_LABEL_DIR.glob("*.json"))
        if len(files) < 2:
            raise ValueError("Training JSON이 최소 2개 필요합니다.")
        if SAMPLE_RATIO is not None:
            count = min(len(files), max(2, round(len(files) * SAMPLE_RATIO)))
            selected = random.Random(SEED).sample(files, count)
        elif MAX_TRAIN_SAMPLES is not None:
            selected = random.Random(SEED).sample(files, min(MAX_TRAIN_SAMPLES, len(files)))
        else:
            selected = files.copy()
        random.Random(SEED).shuffle(selected)
        val_count = max(1, int(len(selected) * INTERNAL_VAL_RATIO))
        val_files, train_files = selected[:val_count], selected[val_count:]
        official_files = sorted(VALIDATION_LABEL_DIR.glob("*.json"))
        if OFFICIAL_VALIDATION_MAX_SAMPLES is not None:
            official_files = official_files[:OFFICIAL_VALIDATION_MAX_SAMPLES]
        if not official_files:
            raise ValueError("공식 Validation JSON이 없습니다.")
        manifest = {
            "selection_config": dataset_selection_config(), "num_training_available": len(files),
            "train_names": [p.name for p in train_files],
            "internal_val_names": [p.name for p in val_files],
            "official_val_names": [p.name for p in official_files],
        }
        digest = hashlib.sha256()
        entries = [("train", p) for p in train_files]
        entries += [("internal_val", p) for p in val_files]
        entries += [("official_val", p) for p in official_files]
        with zipfile.ZipFile(local_archive, "w", compression=zipfile.ZIP_DEFLATED) as packed:
            for role, path in tqdm(entries, desc="JSON 스냅샷 준비"):
                payload = path.read_bytes()
                label = json.loads(payload)
                if not isinstance(label.get("utterances"), list) or "symptom" not in label:
                    raise ValueError(f"utterances 또는 symptom 정답이 없습니다: {path.name}")
                entry = f"{role}/{path.name}"
                digest.update(entry.encode("utf-8") + b"\0" + payload + b"\0")
                packed.writestr(entry, payload)
            manifest["data_digest"] = digest.hexdigest()
            packed.writestr("manifest.json", json.dumps(manifest, ensure_ascii=False))
        publish_file(local_archive, archive)
        print("Drive에 데이터 스냅샷 저장 완료")
    data_dir = LOCAL_WORK_DIR / "labels"
    data_dir.mkdir(exist_ok=True)
    with zipfile.ZipFile(local_archive) as packed:
        packed.extractall(data_dir)
    write_json_result(manifest, "data_manifest.json")
    print(f"선택 {len(manifest['train_names']) + len(manifest['internal_val_names']):,} / "
          f"Training 전체 {manifest['num_training_available']:,}")
    print(f"학습 {len(manifest['train_names']):,}, 내부 검증 {len(manifest['internal_val_names']):,}, "
          f"공식 Validation {len(manifest['official_val_names']):,}")
    return manifest, data_dir

MANIFEST, LOCAL_DATA_DIR = prepare_data_snapshot()

## 7. 전체 층 모델과 통화 단위 평가

In [ ]:
class SymptomClassifier(nn.Module):
    """대화 전사로부터 9종 증상의 logit을 출력."""

    def __init__(self, model_name: str, num_labels: int):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        hidden_size = self.bert.config.hidden_size
        self.dropout = nn.Dropout(0.1)
        self.classifier = nn.Linear(hidden_size, num_labels)

    def forward(
        self,
        input_ids: torch.Tensor,
        attention_mask: torch.Tensor,
    ) -> torch.Tensor:
        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )
        pooled = self.dropout(outputs.last_hidden_state[:, 0])
        return self.classifier(pooled)

def pool_chunk_logits(
    chunk_logits: torch.Tensor,
    conversation_ids: torch.Tensor,
    num_conversations: int,
) -> torch.Tensor:
    """여러 청크의 logit을 증상별 max pooling해 통화 단위로 복원.

    증상이 통화의 어느 한 청크에라도 나타나면 된다는 multi-instance
    learning 가정을 사용한다. sigmoid는 단조함수이므로 logit max pooling은
    확률 max pooling과 같은 청크를 선택하면서 수치적으로 더 안정적이다.
    """
    pooled = []
    for conversation_index in range(num_conversations):
        mask = conversation_ids == conversation_index
        if not torch.any(mask):
            raise ValueError(f"통화 {conversation_index}에 대응하는 청크가 없습니다.")
        pooled.append(chunk_logits[mask].amax(dim=0))
    return torch.stack(pooled, dim=0)

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

class ConversationDataset(Dataset):
    def __init__(self, folder, names, tokenizer):
        self.folder, self.names, self.tokenizer = folder, names, tokenizer

    def __len__(self):
        return len(self.names)

    def __getitem__(self, index):
        name = self.names[index]
        label = json.loads((self.folder / name).read_text(encoding="utf-8"))
        encoded = tokenize_dialogue_chunks(
            self.tokenizer, label["utterances"], MAX_LENGTH, LONG_TURN_OVERLAP,
        )
        symptoms = set(label["symptom"])
        encoded["labels"] = torch.tensor([int(s in symptoms) for s in TARGET_SYMPTOMS], dtype=torch.float32)
        encoded["name"] = name
        return encoded

def collate_conversations(samples):
    return {
        "input_ids": torch.cat([s["input_ids"] for s in samples]),
        "attention_mask": torch.cat([s["attention_mask"] for s in samples]),
        "conversation_ids": torch.cat([
            torch.full((s["input_ids"].shape[0],), i, dtype=torch.long)
            for i, s in enumerate(samples)
        ]),
        "labels": torch.stack([s["labels"] for s in samples]),
    }

@torch.no_grad()
def evaluate_dataset(model, dataset, description):
    model.eval()
    truths, predictions, rows = [], [], []
    for index in tqdm(range(len(dataset)), desc=description):
        sample = dataset[index]
        combined = None
        for start in range(0, len(sample["input_ids"]), EVAL_CHUNK_BATCH):
            logits = model(
                sample["input_ids"][start:start + EVAL_CHUNK_BATCH].to(DEVICE),
                sample["attention_mask"][start:start + EVAL_CHUNK_BATCH].to(DEVICE),
            ).max(dim=0).values
            combined = logits if combined is None else torch.maximum(combined, logits)
        prediction = (combined.sigmoid() > THRESHOLD).to(torch.int8).cpu().numpy()
        truths.append(sample["labels"].numpy().astype(np.int8))
        predictions.append(prediction)
        rows.append({
            "label file name": sample["name"],
            "symptom": str([name for name, flag in zip(TARGET_SYMPTOMS, prediction) if flag]),
        })
    return compute_multilabel_metrics(np.stack(truths), np.stack(predictions), TARGET_SYMPTOMS), pd.DataFrame(rows)

"""학습 통화의 라벨 빈도로 양성 손실 가중치를 계산한다."""

import json
import math
from pathlib import Path

import torch


def compute_pos_weights(label_files, target_symptoms, scale=1.0, max_weight=None):
    """내부 검증을 제외한 통화 파일만 전달한다. 청크 수는 세지 않는다."""
    if not math.isfinite(scale) or scale < 0:
        raise ValueError("pos_weight scale은 유한한 0 이상의 값이어야 합니다.")
    if max_weight is not None and (not math.isfinite(max_weight) or max_weight < 1):
        raise ValueError("pos_weight 상한은 유한한 1 이상의 값이어야 합니다.")
    if not label_files:
        raise ValueError("가중치를 계산할 학습 데이터가 없습니다.")
    positives = torch.zeros(len(target_symptoms), dtype=torch.float32)
    for path in label_files:
        label = json.loads(Path(path).read_text(encoding="utf-8"))
        symptoms = set(label.get("symptom", []))
        positives += torch.tensor([int(s in symptoms) for s in target_symptoms])
    negatives = len(label_files) - positives
    # 양성이 없는 라벨은 추정 불가. 양성 다수 라벨도 손실을 낮추지 않는다.
    ratios = (negatives / positives.clamp_min(1)).clamp_min(1)
    ratios = torch.where(positives > 0, ratios, torch.ones_like(ratios))
    weights = 1 + scale * (ratios - 1)
    if max_weight is not None:
        weights = weights.clamp_max(max_weight)
    details = {
        "name": "bce_pos_weight", "scale": scale, "max_weight": max_weight,
        "target_symptoms": list(target_symptoms), "num_train": len(label_files),
        "positive_counts": positives.to(torch.int64).tolist(),
        "negative_counts": negatives.to(torch.int64).tolist(),
        "pos_weight": weights.tolist(),
    }
    return weights, details


## 8. 고정 임계값 0.5로 내부·공식 Validation 평가
이 셀은 학습을 실행하지 않습니다. 원래의 학습 셀을 다시 붙여넣지 마세요.
`best.pt`의 데이터 digest, 청크 설정, 증상 순서를 검사한 뒤 평가합니다.
내부 검증과 공식 Validation은 각각 한 번 순전파하고 모든 증상에 임계값 0.5를 적용합니다.


In [ ]:
"""모든 증상에 고정 임계값 0.5를 적용한다."""

import numpy as np


def validate_threshold_data(truths, scores, label_names):
    truths, scores = np.asarray(truths), np.asarray(scores, dtype=float)
    if truths.shape != scores.shape or truths.ndim != 2:
        raise ValueError("정답과 점수는 동일한 (통화 수, 증상 수) 배열이어야 합니다.")
    if not len(truths) or truths.shape[1] != len(label_names):
        raise ValueError("검증 통화가 없거나 증상 순서의 길이가 다릅니다.")
    if not np.isin(truths, [0, 1]).all():
        raise ValueError("정답은 0/1이어야 합니다.")
    if not np.isfinite(scores).all() or np.any((scores < 0) | (scores > 1)):
        raise ValueError("sigmoid 점수는 유한한 0~1 값이어야 합니다.")
    return truths.astype(np.int8), scores


def fixed_threshold_result(truths, scores, label_names, metric_fn):
    """튜닝 없이 모든 라벨에 0.5를 적용한 결과를 반환한다."""
    truths, scores = validate_threshold_data(truths, scores, label_names)
    thresholds = [THRESHOLD] * len(label_names)
    metrics = metric_fn(truths, scores > THRESHOLD, label_names)
    return {
        "objective": "fixed_threshold", "method": THRESHOLD_POLICY,
        "thresholds": thresholds, "target_symptoms": list(label_names),
        "num_internal_validation": len(truths),
        "selection_source": "mentor_policy",
        "internal_candidates": [
            {"method": THRESHOLD_POLICY, "thresholds": thresholds, "metrics": metrics}
        ],
        "global_search": [],
    }


In [ ]:
@torch.no_grad()
def collect_threshold_scores(model, dataset, description):
    model.eval()
    truths, scores, names = [], [], []
    for index in tqdm(range(len(dataset)), desc=description):
        sample = dataset[index]
        combined = None
        for start in range(0, len(sample["input_ids"]), EVAL_CHUNK_BATCH):
            logits = model(
                sample["input_ids"][start:start + EVAL_CHUNK_BATCH].to(DEVICE),
                sample["attention_mask"][start:start + EVAL_CHUNK_BATCH].to(DEVICE),
            ).max(dim=0).values
            combined = logits if combined is None else torch.maximum(combined, logits)
        truths.append(sample["labels"].numpy().astype(np.int8))
        scores.append(combined.sigmoid().float().cpu().numpy())
        names.append(sample["name"])
    return np.stack(truths), np.stack(scores), names

REVIEW_DIR = DRIVE_RUN_DIR / "threshold_review"
REVIEW_DIR.mkdir(parents=True, exist_ok=True)

def save_review_json(payload, filename):
    temporary = LOCAL_WORK_DIR / filename
    temporary.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
    publish_file(temporary, REVIEW_DIR / filename)

best_source = DRIVE_RUN_DIR / "best.pt"
if not best_source.is_file():
    raise FileNotFoundError(f"저장된 모델이 없습니다: {best_source}")
best_local = LOCAL_WORK_DIR / "threshold_review_best.pt"
shutil.copyfile(best_source, best_local)
best = torch.load(best_local, map_location="cpu", weights_only=False)
if best["data_digest"] != MANIFEST["data_digest"]:
    raise ValueError("모델과 데이터 스냅샷이 다릅니다. 원래 RUN_NAME과 설정을 사용하세요.")
if best["chunk_strategy"] != CHUNK_STRATEGY or best["target_symptoms"] != TARGET_SYMPTOMS:
    raise ValueError("평가 모델의 청크 방식 또는 증상 순서가 다릅니다.")
if best["max_length"] != MAX_LENGTH or best["long_turn_overlap"] != LONG_TURN_OVERLAP:
    raise ValueError("학습 당시의 MAX_LENGTH와 LONG_TURN_OVERLAP을 사용하세요.")
evaluation_tokenizer = AutoTokenizer.from_pretrained(best["model_name"])
evaluation_model = SymptomClassifier(best["model_name"], len(TARGET_SYMPTOMS)).to(DEVICE)
evaluation_model.load_state_dict(best["model_state_dict"])
best_epoch = best["epoch"]
checkpoint_loss = best.get("loss_config", {})
del best
gc.collect()
print("저장 모델의 가중치 설정:", checkpoint_loss)

internal = ConversationDataset(LOCAL_DATA_DIR / "internal_val", MANIFEST["internal_val_names"], evaluation_tokenizer)
internal_truths, internal_scores, internal_names = collect_threshold_scores(evaluation_model, internal, "내부 검증 점수")
SELECTION = fixed_threshold_result(internal_truths, internal_scores, TARGET_SYMPTOMS, compute_multilabel_metrics)
SELECTION.update({"data_digest": MANIFEST["data_digest"], "best_epoch": best_epoch, "checkpoint_loss": checkpoint_loss})
save_review_json(SELECTION, "threshold_selection.json")
metric_columns = ["macro_f1", "macro_precision", "macro_recall", "subset_accuracy", "label_accuracy", "hamming_loss"]
print("내부 검증 결과 (모든 증상 임계값 0.5)")
display(pd.DataFrame([{"method": c["method"], **{key: c["metrics"][key] for key in metric_columns}} for c in SELECTION["internal_candidates"]]))
print("임계값 정책:", SELECTION["method"])
display(pd.DataFrame({"증상": TARGET_SYMPTOMS, "고정 임계값": SELECTION["thresholds"]}))

official = ConversationDataset(LOCAL_DATA_DIR / "official_val", MANIFEST["official_val_names"], evaluation_tokenizer)
official_truths, official_scores, official_names = collect_threshold_scores(evaluation_model, official, "공식 Validation 점수")
BASELINE_METRICS = compute_multilabel_metrics(official_truths, official_scores > THRESHOLD, TARGET_SYMPTOMS)
selected_predictions = official_scores > THRESHOLD
SELECTED_METRICS = compute_multilabel_metrics(official_truths, selected_predictions, TARGET_SYMPTOMS)
COMPARISON = {"fixed_0.5": BASELINE_METRICS, "internal_selected": SELECTED_METRICS, "selection_method": SELECTION["method"], "objective": "fixed_threshold", "thresholds": SELECTION["thresholds"], "data_digest": MANIFEST["data_digest"], "best_epoch": best_epoch}
save_review_json(COMPARISON, "official_threshold_comparison.json")
print("공식 Validation 결과 (모든 증상 임계값 0.5)")
display(pd.DataFrame([{"method": name, **{key: metrics[key] for key in metric_columns}} for name, metrics in [("fixed_0.5", BASELINE_METRICS), ("internal_selected", SELECTED_METRICS)]]))
print("증상별 오탐·미탐 비교")
display(pd.DataFrame([{"증상": symptom, "임계값": SELECTION["thresholds"][i], "기존 FP": BASELINE_METRICS["per_label"][symptom]["fp"], "선택 FP": SELECTED_METRICS["per_label"][symptom]["fp"], "기존 FN": BASELINE_METRICS["per_label"][symptom]["fn"], "선택 FN": SELECTED_METRICS["per_label"][symptom]["fn"], **{key: SELECTED_METRICS["per_label"][symptom][key] for key in ("precision", "recall", "f1")}} for i, symptom in enumerate(TARGET_SYMPTOMS)]))
SUBMISSION = pd.DataFrame({"label file name": official_names, "symptom": [str([name for name, flag in zip(TARGET_SYMPTOMS, row) if flag]) for row in selected_predictions]})
csv_local = LOCAL_WORK_DIR / "mission3_threshold_review.csv"
SUBMISSION.to_csv(csv_local, index=False, encoding="utf-8-sig")
publish_file(csv_local, REVIEW_DIR / csv_local.name)
# 저장한 점수로 임계값 비교를 재현할 수 있습니다.
scores_local = LOCAL_WORK_DIR / "threshold_review_scores.npz"
np.savez_compressed(scores_local, internal_truths=internal_truths, internal_scores=internal_scores, internal_names=np.asarray(internal_names), official_truths=official_truths, official_scores=official_scores, official_names=np.asarray(official_names), target_symptoms=np.asarray(TARGET_SYMPTOMS))
publish_file(scores_local, REVIEW_DIR / scores_local.name)
print("결과 폴더:", REVIEW_DIR)
